# Making Document Extraction Worth Automating — Colab runner

Everything is fetched from open sources, so anyone can rerun the project:

| What | Where | Access |
|---|---|---|
| Code | GitHub (`CODE_REPO`) | public, cloned below |
| Results (runs, predictions, metrics, logs) | Hugging Face Hub dataset (`RESULTS_REPO`) | public download; pushing needs the owner's token |
| CORD, WildReceipt | official public sources | no account |
| DocILE | official source | each user needs their own free token (docile.rossum.ai) |

**To reproduce the published results without a GPU**, run the first cell, then the "prepare" cells of each dataset and the routing/summary cells: finished runs are pulled from `RESULTS_REPO` and skipped.
**To run new experiments**, set `RESULTS_REPO` to your own repo and use **Save results** after each section.

Training jobs run several at a time on one GPU (`PARALLEL`) and skip anything already finished, so any cell can be re-run safely.

In [ ]:
CODE_REPO = 'https://github.com/<github-user>/doc-extraction'   # code
RESULTS_REPO = '<hf-user>/doc-extraction-results'               # Hugging Face dataset repo for results

from pathlib import Path
ROOT = Path('/content/doc-extraction')
if not ROOT.exists():
    !git clone -q {CODE_REPO} {ROOT}
%cd {ROOT}
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -r requirements.txt
!python -m pytest tests -q
!python -m docext.hub pull {RESULTS_REPO}

In [ ]:
import json, subprocess, time
from pathlib import Path
from docext.train import TrainConfig

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
PARALLEL = 3 if 'A100' in GPU else 2   # trainings sharing the GPU; lower it if you see CUDA out-of-memory

def job(data, eval_splits, seed=0, **settings):
    """One training run: its folder name and command-line arguments."""
    cfg = TrainConfig(data=data, eval_splits=list(eval_splits), seed=seed, **settings)
    args = ['--data', data, '--eval-splits', *eval_splits, '--seed', str(seed)]
    for key, value in settings.items():
        args += ['--' + key.replace('_', '-'), str(value)]
    return cfg.run_name(), args

def launch(jobs, parallel=PARALLEL):
    """Run jobs `parallel` at a time, skipping finished runs. Blocks until all end."""
    todo = [(n, a) for n, a in jobs if not (Path('runs') / n / 'metrics.json').exists()]
    print(f'{len(jobs) - len(todo)} already done, {len(todo)} to run, {parallel} at a time')
    Path('logs').mkdir(exist_ok=True)
    running = []
    while todo or running:
        while todo and len(running) < parallel:
            name, args = todo.pop(0)
            log = open(f'logs/{name}.log', 'w')
            proc = subprocess.Popen(['python', '-m', 'docext.train', *args], stdout=log, stderr=subprocess.STDOUT)
            running.append((name, proc, log))
        time.sleep(15)
        for item in list(running):
            name, proc, log = item
            if proc.poll() is not None:
                log.close()
                running.remove(item)
                print('finished' if proc.returncode == 0 else f'FAILED, see logs/{name}.log:', name)

def show(run_name, split):
    """Entity F1 of one run: overall, per x-shot subset and the many-minus-zero gap."""
    m = json.loads((Path('runs') / run_name / 'metrics.json').read_text())[split]
    return {k: round(v['f1'] if isinstance(v, dict) else v, 4) for k, v in m.items() if k != 'docile_kile'}

print(GPU, '| parallel jobs:', PARALLEL)

## 1. CORD: data, noise floor, template-disjoint split

In [ ]:
!python -m docext.prepare cord --out data
!python -m docext.prepare cord-disjoint --out data
!python -m docext.analyse data/cord/train.pkl --model layoutlmv3

## 2. CORD: baselines on both splits, then tuning on validation

Δ on CORD = F1(official test) − F1(disjoint test). Tuning (jitter magnitude, key swap, structural weights) uses the disjoint split's validation set.
While this runs, check progress in a second cell with `!tail -n 2 logs/*.log`.

In [ ]:
CORD = ['validation', 'test']
jobs = [job('data/cord', CORD), job('data/cord_disjoint', CORD)]
jobs += [job('data/cord_disjoint', CORD, jitter=j) for j in (0.01, 0.03, 0.05, 0.1)]
jobs += [job('data/cord_disjoint', CORD, key_swap=0.5)]
jobs += [job('data/cord_disjoint', CORD, lambda_arith=w, lambda_card=w) for w in (0.1, 0.3, 1.0)]
launch(jobs)
!python -m docext.summarise runs --split validation

## 3. CORD routing (no GPU needed)

In [ ]:
for d in sorted(Path('runs').glob('cord*')):
    subprocess.run(['python', '-m', 'docext.routing', str(d), '--split', 'test', '--target', '0.95'], check=True)

## 4. WildReceipt: unseen-template benchmark (public, no token)

1,268 training and 472 test receipts; the test templates never appear in training. A validation set of *seen* templates is held out from training, so Δ = F1(validation) − F1(test).
`split_report.json` shows `test_replication`: near 0 means our template clustering agrees with the authors' unseen-template split.

In [ ]:
!wget -q -nc https://download.openmmlab.com/mmocr/data/wildreceipt.tar -P data
!mkdir -p data/wildreceipt_raw && tar -xf data/wildreceipt.tar -C data/wildreceipt_raw --skip-old-files
WR_RAW = next(Path('data/wildreceipt_raw').rglob('class_list.txt')).parent
!python -m docext.prepare wildreceipt --wildreceipt-path {WR_RAW} --out data
!python -m docext.analyse data/wildreceipt/train.pkl --model layoutlmv3

## 5. WildReceipt experiments, in priority order

Fill `BEST` from the CORD tuning table. Runs are CORD-sized, so 3 at a time on an A100.

| Tier | Runs | Answers |
|---|---|---|
| 1 | every arm, seed 0 | does each method shrink Δ? |
| 2 | every arm, seeds 1–2 | is the change larger than seed noise? |
| 3 | RoBERTa and BROS, baseline and best, 3 seeds | does the finding depend on architecture? |

In [ ]:
WR = ['validation', 'test']
BEST = dict(jitter=0.05, key_swap=0.5, lambda_arith=0.3, lambda_card=0.3)
ARMS = {
    'baseline': {},
    'jitter': {'jitter': BEST['jitter']},
    'key_swap': {'key_swap': BEST['key_swap']},
    'consistency': {'lambda_arith': BEST['lambda_arith'], 'lambda_card': BEST['lambda_card']},
    'all': BEST,
}
tier1 = [job('data/wildreceipt', WR, **a) for a in ARMS.values()]
tier2 = [job('data/wildreceipt', WR, seed=s, **a) for s in (1, 2) for a in ARMS.values()]
tier3 = [job('data/wildreceipt', WR, seed=s, model=m, **a)
         for m in ('roberta', 'bros') for a in ({}, BEST) for s in (0, 1, 2)]
launch(tier1)

In [ ]:
launch(tier2 + tier3)
!python -m docext.summarise runs --split validation --csv results_wr_validation.csv
!python -m docext.summarise runs --split test --csv results_wr_test.csv

In [ ]:
# routing on the unseen-template test set
for d in sorted(Path('runs').glob('wildreceipt_*')):
    subprocess.run(['python', '-m', 'docext.routing', str(d), '--split', 'test', '--target', '0.90'], check=True)

## Inspect one routing result

In [ ]:
from IPython.display import Image, display
run_name = 'wildreceipt_layoutlmv3_j0.0_k0.0_a0.0_c0.0_s0'   # any finished run
print(json.dumps(json.loads(Path(f'runs/{run_name}/routing_test/summary.json').read_text()), indent=1))
display(Image(f'runs/{run_name}/routing_test/risk_coverage.png'))

## 6. DocILE: download, convert, noise floor

The zip (several GB) is kept on Drive, so it downloads once. Each session unzips it to the fast local disk (`/content/docile_raw`): Drive is slow with thousands of small files, and the official evaluator reads from there.
The token is typed into a hidden prompt, so it never appears in the notebook.

In [ ]:
DOCILE_RAW = '/content/docile_raw'
if not Path('data/labeled-trainval.zip').exists():
    from getpass import getpass
    token = getpass('DocILE token: ')
    !git clone -q --depth 1 https://github.com/rossumai/docile.git /content/docile_repo
    !bash /content/docile_repo/download_dataset.sh {token} labeled-trainval data
!mkdir -p {DOCILE_RAW} && unzip -qn data/labeled-trainval.zip -d {DOCILE_RAW}
if not Path('data/docile/val.pkl').exists():
    !python -m docext.prepare docile --docile-path {DOCILE_RAW} --out data
!python -m docext.analyse data/docile/train.pkl --model layoutlmv3

## 7. DocILE runs, in priority order

DocILE runs are larger, so 2 at a time. Watch the `seconds` per epoch in the first logs and adjust `EPOCHS` if needed.
Fill `BEST` from the CORD tuning table.

| Tier | Runs | Answers |
|---|---|---|
| 1 | every arm, seed 0 | does each method change Δ? |
| 2 | baseline and best, seeds 1–2 | is the gain larger than seed noise? |
| 3 | RoBERTa and BROS, baseline and best | does the finding depend on architecture? |

In [ ]:
EPOCHS = 6
DOC = ['val']
BEST = dict(jitter=0.05, key_swap=0.5, lambda_arith=0.3, lambda_card=0.3)
ARMS = {
    'baseline': {},
    'jitter': {'jitter': BEST['jitter']},
    'key_swap': {'key_swap': BEST['key_swap']},
    'consistency': {'lambda_arith': BEST['lambda_arith'], 'lambda_card': BEST['lambda_card']},
    'all': BEST,
}
tier1 = [job('data/docile', DOC, epochs=EPOCHS, **s) for s in ARMS.values()]
launch(tier1, parallel=2)

## 8. DocILE tiers 2 and 3

In [ ]:
tier2 = [job('data/docile', DOC, seed=s, epochs=EPOCHS, **a) for s in (1, 2) for a in ({}, BEST)]
tier3 = [job('data/docile', DOC, model=m, epochs=EPOCHS, **a) for m in ('roberta', 'bros') for a in ({}, BEST)]
launch(tier2 + tier3, parallel=2)

## 9. DocILE official scores, routing, summary

Run section 6's first cell again after a disconnect, so the data is back in `/content/docile_raw`.

In [ ]:
for d in sorted(Path('runs').glob('docile_*')):
    subprocess.run(['python', '-m', 'docext.evaluate', str(d), '--docile-path', DOCILE_RAW],
                   check=True, capture_output=True)
    subprocess.run(['python', '-m', 'docext.routing', str(d), '--split', 'val', '--target', '0.95'], check=True)
!python -m docext.summarise runs --split val --csv results_docile.csv
!python -m docext.summarise runs --split test --csv results_cord.csv

## Save results

Run after each section. Uploads runs, logs and reports to `RESULTS_REPO` (public unless you add `--private`). The first time in a session, paste a **write** token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
!python -m docext.hub push {RESULTS_REPO}